# 2026 COMP90042 Project — AutoDL 版
*Make sure you change the file name with your group id.*

> **此文件为 AutoDL 适配版**。主交付物仍是 `notebooks/notebook.ipynb`（Colab T4 基准版）。
> 本版本仅用于在 AutoDL GPU 实例上进行更快速的训练迭代；结构与原版保持一致，
> 路径、镜像源、显存预算已针对 AutoDL 环境调整。

# Readme

**Group:** _<fill in>_

**System overview.** Hybrid RAG fact-checker for climate claims:

1. **Stage 0** — SFT data construction (tagging × domain × difficulty, hash split, ms-swift format + curriculum).
2. **Stage 1** — Hybrid retrieval: BM25 (`bm25s`) + dense `BAAI/bge-m3` + fusion (0.3/0.7) + cross-encoder rerank + rule reorder. Label-conditioned `k`.
3. **Stage 2** — Query rewriting (synonym, sub-claim split, HyDE).
4. **Stage 3** — SFT on Qwen3.5-4B (text-only, ModelScope) via ms-swift. QLoRA 4-bit on RTX 3090; optional full bf16 on A100.
5. **Stage 4** — DPO preference alignment (mistakes mined from `dev_holdout`).
6. **Stage 5** — Self-consistency inference (5 samples @ T=0.7, majority vote).
7. **Stage 6** — Per-bucket ablation + error attribution on `diag_test`.

**Reproducibility note.** Assignment requires Colab T4 (15 GB VRAM) reproducibility.
This AutoDL version targets RTX 3090/4090 (24 GB) for faster iteration but
keeps the same conservative QLoRA settings so checkpoints are interchangeable.

---
### Section status legend
- ✅ **Verified locally (dry-run)** — CPU-only, no GPU needed.
- 🧪 **Stub-validated locally** — wiring tested, real numbers need GPU.
- ⏳ **Requires GPU** — heavy compute (download / indexing / training / inference).

## AutoDL 一次性初始化指南

> 在 AutoDL 控制台完成以下步骤，之后每次开机只需从 **Setup** 节开始跑 notebook。

### 推荐实例配置
| 项目 | 推荐 | 最低 |
|---|---|---|
| GPU | RTX 4090 / RTX 3090 (24 GB) | RTX 3080 / T4 (≥10 GB) |
| RAM | 60 GB+ | 30 GB |
| 数据盘 (`/root/autodl-tmp`) | 100 GB | 50 GB |
| 系统盘 (`/root`) | 50 GB | 30 GB |

### 关于磁盘选择 — 重要

AutoDL 有两块盘，**都不是真正"临时"**（关机重启都保留，只有"释放实例"时才丢）：

| 路径 | 容量 | 速度 | 用途 |
|---|---|---|---|
| `/root/`（系统盘）| ~50 GB | 较慢 | 默认 home，空间紧张 |
| `/root/autodl-tmp/`（数据盘）| 50-200 GB | NVMe 快 | 推荐放代码 + 数据 + 模型 |

**推荐：整个 `Assignment3/` 项目（含 notebook）直接放在 `/root/autodl-tmp/` 下**，
省掉软链接，I/O 全走 NVMe，模型/索引/checkpoint 不挤占系统盘。

### 第一步：进入数据盘 + 上传代码
```bash
cd /root/autodl-tmp

# 方式 A — git clone（推荐）
git clone https://github.com/<your_org>/Assignment3.git Assignment3

# 方式 B — AutoDL 文件管理器
# 控制台 → 实例 → 文件管理 → 上传 zip → 解压到 /root/autodl-tmp/Assignment3/
```

### 第二步：放置数据
```bash
# evidence.json (174 MB) + claim 文件 → 直接放在项目内 data/
mkdir -p /root/autodl-tmp/Assignment3/data
# 用文件管理器上传，或从对象存储 wget 拉下来
```

### 第三步：打开 notebook
```
JupyterLab → /root/autodl-tmp/Assignment3/notebooks/notebook_autodl.ipynb
```

### 推荐目录结构
```
/root/autodl-tmp/
├── Assignment3/                    ← git clone 到这里
│   ├── notebooks/
│   │   ├── notebook.ipynb          ← Colab 版（最终提交用）
│   │   └── notebook_autodl.ipynb   ← 本文件
│   ├── src/
│   ├── data/                       ← evidence.json 直接放这（无软链接）
│   └── outputs/                    ← 切分 / SFT jsonl 等小产物
└── nlp_a3_cache/                   ← 大文件缓存（与代码同盘但分目录）
    ├── model_cache/                ← Qwen3.5-4B (~8 GB)
    ├── bm25_index/                 (~200 MB)
    ├── dense_index/                (~5 GB FAISS)
    ├── sft-out/                    (~500 MB LoRA)
    └── dpo-out/                    (~500 MB LoRA)
```

### 兼容旧布局
如果你已经把代码放在 `/root/Assignment3/`，setup-1 cell 会自动探测识别，
data 软链接到 `/root/autodl-tmp/data/` 也能正常工作。无需迁移。

### 重要提示
- `/root/autodl-tmp/` 关机重启保留，**仅"释放实例"时丢失**。重要 checkpoint 请额外备份。
- 镜像内 PyTorch/CUDA 已预装，notebook 安装步骤只装应用层依赖（< 2 min）。
- 每次重启实例（不释放）变量丢失，但 `/root/autodl-tmp/` 文件保留。
  从 Setup cell 顺序重跑即可，cache 命中后 < 5 min 恢复到可推理状态。

## Setup
检测 AutoDL / Colab / 本地环境；设定路径；安装依赖（国内镜像）；检测 GPU；设随机种子。

In [ ]:
import os, sys
from pathlib import Path

# ── 环境检测 ──────────────────────────────────────────────────────────
IS_COLAB  = "google.colab" in sys.modules
IS_AUTODL = os.path.exists("/root/autodl-tmp") and not IS_COLAB


def _find_project_root() -> Path:
    """
    优先级：
      1. PROJECT_ROOT 环境变量
      2. 从 notebook 当前 CWD 向上找 `src/paths.py`（CWD 失效则跳过）
      3. 常见预设位置（autodl-tmp/Assignment3 → /root/Assignment3）
    """
    env = os.environ.get("PROJECT_ROOT")
    if env and Path(env).exists():
        return Path(env)

    # 沿 CWD 向上找仓库根；CWD 可能已失效（kernel 留在被删/改名的目录），try 包住
    try:
        cur = Path.cwd().resolve()
        for parent in [cur, *cur.parents]:
            if (parent / "src" / "paths.py").exists():
                return parent
    except (FileNotFoundError, OSError):
        pass  # CWD 失效，落到兜底

    # 兜底：常见 AutoDL 布局
    for cand in ("/root/autodl-tmp/Assignment3", "/root/Assignment3"):
        if (Path(cand) / "src" / "paths.py").exists():
            return Path(cand)

    raise FileNotFoundError(
        "找不到项目根目录。请确认下列任一条件：\n"
        "  • 设置环境变量 PROJECT_ROOT 指向项目根（含 src/paths.py），或\n"
        "  • 项目放在 /root/autodl-tmp/Assignment3/ 且其下有 src/paths.py，或\n"
        "  • notebook 在仓库 notebooks/ 子目录下打开"
    )


if IS_AUTODL:
    AUTODL_TMP   = "/root/autodl-tmp"
    PROJECT_ROOT = str(_find_project_root())
    CACHE_ROOT   = f"{AUTODL_TMP}/nlp_a3_cache"

elif IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/NLP_Assignment3/Assignment3"
    CACHE_ROOT   = PROJECT_ROOT + "/outputs"

else:
    PROJECT_ROOT = str(_find_project_root())
    CACHE_ROOT   = PROJECT_ROOT + "/outputs"

# 导出给 src/paths.py 及后续 cell 使用
os.environ["PROJECT_ROOT"] = PROJECT_ROOT
os.environ["CACHE_ROOT"]   = CACHE_ROOT
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)   # 修复 kernel CWD（如果之前失效）

Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT, "outputs").mkdir(parents=True, exist_ok=True)

ENV = "autodl" if IS_AUTODL else "colab" if IS_COLAB else "local"
for k, v in [("ENV", ENV), ("PROJECT_ROOT", PROJECT_ROOT), ("CACHE_ROOT", CACHE_ROOT)]:
    print(f"{k:14s}= {v}")
print("CWD           =", os.getcwd())

In [ ]:
# ── 安装依赖（国内镜像）──────────────────────────────────────────────
# AutoDL 镜像已预装 PyTorch/CUDA；此处只装应用层包，约 1-2 分钟。
if IS_AUTODL or IS_COLAB:
    MIRROR = "https://pypi.tuna.tsinghua.edu.cn/simple"
    _pkgs = (
        "ms-swift modelscope transformers peft trl "
        "bitsandbytes accelerate sentence-transformers "
        "bm25s pystemmer faiss-cpu spacy nltk"
    )
    os.system(f"pip install -q -i {MIRROR} {_pkgs}")
    os.system("python -m spacy download en_core_web_sm -q")
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    print("deps installed / verified")
else:
    print("local env — skip pip install (run manually if needed)")

In [ ]:
# ── GPU 检测 + 自适应训练参数 ────────────────────────────────────────
import torch, gc

VRAM_GB = 0
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    VRAM_GB = _p.total_memory // 2 ** 30
    print(f"GPU  : {_p.name}  |  VRAM: {VRAM_GB} GB")
    _free_ram = os.popen("free -g 2>/dev/null | awk '/^Mem/{print $7}'").read().strip()
    print(f"RAM  : {_free_ram or '?'} GB available")
else:
    print("No GPU — Stage 0 CPU-only prep is fine without one.")

# 根据显存自动选择训练超参（有效 batch size 保持 16）
if VRAM_GB >= 40:           # A100 / A800 (40 GB+)  — 可关 4-bit
    SFT_BS, SFT_GA, USE_4BIT = 4, 4, False
    DENSE_BATCH = 128
elif VRAM_GB >= 24:         # RTX 3090 / RTX 4090 (24 GB)  ← AutoDL 主力
    SFT_BS, SFT_GA, USE_4BIT = 2, 8, True
    DENSE_BATCH = 64
else:                        # T4 / RTX 3080 (≤16 GB)  — Colab 基准设置
    SFT_BS, SFT_GA, USE_4BIT = 1, 16, True
    DENSE_BATCH = 32

MAX_LEN    = 1024
QUANT_FLAG = "--quantization_bit 4 --bnb_4bit_compute_dtype bfloat16" if USE_4BIT else ""

print(f"\nAdaptive SFT : BS={SFT_BS}  grad_accum={SFT_GA}  eff_bs={SFT_BS*SFT_GA}  4bit={USE_4BIT}")
print(f"Dense batch  : {DENSE_BATCH}")

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass
print("seed =", SEED)

### 快速恢复（重启实例后运行）

实例重启后变量清空，但 `/root/autodl-tmp/` 文件保留。
重跑上面 4 个 Setup cell 后，用下面的 sanity check 确认缓存完整性，
再按需跳转到对应 section 重载变量（无需重新 build）。

```python
# Sanity check — 粘贴到新 cell 跑
from pathlib import Path
CACHE = Path(CACHE_ROOT)
checks = {
    "evidence.json":       Path(PROJECT_ROOT, "data", "evidence.json"),
    "bm25_index":          CACHE / "bm25_index" / "ev_ids.txt",
    "faiss.index":         CACHE / "dense_index" / "faiss.index",
    "sft_train_v1.jsonl":  Path(PROJECT_ROOT, "outputs", "sft_data", "sft_train_v1.jsonl"),
    "sft checkpoint":      CACHE / "sft-out" / "checkpoint-final",
    "dpo checkpoint":      CACHE / "dpo-out" / "checkpoint-final",
}
for name, p in checks.items():
    print(f"{'✓' if p.exists() else '✗'}  {name:28s}  {p}")
```

**按缺失 ✗ 重跑对应 cell（命中 cache 总计 < 5 min）：**

| 缺失 | 重跑 cell | 耗时 |
|---|---|---|
| `evidence` 变量 | 1.1 load | ~20 s |
| `bm25` | 2.1 BM25 | < 5 s |
| `dense` | 2.2 Dense | < 30 s |
| `pipeline_zero_shot` | 2.3 Pipeline | ~15 s |
| `MODEL_DIR` | 2.5 download | < 5 s (cache hit) |
| `base_model` | 3.5a | ~20 s |

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

### 1.1 Load claims + run EDA &nbsp;&nbsp;✅ verified locally

**Findings recorded in `outputs/eda/eda_report.md`:**
- train 1,228 / dev 154 / test 153 claims; corpus 1,208,827 evidence passages.
- Claim length: median 19 tokens, max 67. Evidence: median 18 tokens, max 479.
- Label imbalance: SUPPORTS 42% / NEI 31% / REFUTES 16% / DISPUTED 10%.
- **Critical prior:** every NOT_ENOUGH_INFO claim has *exactly 5* gold evidences → drives label-conditioned `k`.

In [ ]:
from src.data_io import load_train, load_dev, load_test_unlabelled, load_evidence
from src.eda import build_report

train, dev, test = load_train(), load_dev(), load_test_unlabelled()
evidence = load_evidence()   # 174 MB, ~1.2 M passages; takes ~20 s
print(f"train={len(train)}  dev={len(dev)}  test={len(test)}  corpus={len(evidence):,}")

report_path = build_report()
print("EDA →", report_path)

### 1.2 Three-axis tagging (scenario × domain × difficulty) &nbsp;&nbsp;✅ verified locally

Tags serve as error-attribution handles. After SFT we slice `diag_test` errors by these axes
to identify the worst-performing bucket and motivate the next data-centric iteration.

- **Scenario** (7 classes, label-driven)
- **Domain** (8 climate-science classes via keyword + NER)
- **Difficulty** (3 levels via heuristic score: claim length + n_evidence + per-label prior)

In [ ]:
from src.stage0_tag import run as run_tagging
tagged_path, dist_path = run_tagging()
print("jsonl:", tagged_path)
print("dist :", dist_path)
print(dist_path.read_text(encoding="utf-8"))

### 1.3 Hash split (train_split / dev_holdout / diag_test) + leakage check &nbsp;&nbsp;✅ verified locally

`md5(salt || claim_id) % 10` → 0–7 train_split (~80%), 8 dev_holdout (~10%), 9 diag_test (~10%).
Six pairwise intersection assertions guarantee no leakage into official dev/test files.

In [ ]:
from src.splits import run as run_splits
split_paths = run_splits()
print(split_paths["summary"].read_text(encoding="utf-8"))

### 1.4 Build ms-swift SFT dataset &nbsp;&nbsp;✅ verified locally

Each `train_split` row → `{system, query, response}`. Hard-negative augmentation
adds random non-gold evidences relabeled NEI. Curriculum: easy → medium → hard within each epoch.

In [ ]:
from pathlib import Path
from src.data_io import read_jsonl, write_jsonl
from src.paths import SPLITS_DIR, SFT_DIR
from src.sft_dataset import build_dataset

tagged_train = list(read_jsonl(SPLITS_DIR / "train_split.jsonl"))
sft_train = build_dataset(
    tagged_train, evidence, k=5, pad_with_random=True, n_hard_neg=1, seed=SEED
)
out = SFT_DIR / "sft_train_v1.jsonl"
write_jsonl(sft_train, out)
print(f"wrote {out}: {len(sft_train)} records")

import json
print(json.dumps(sft_train[0], indent=2, ensure_ascii=False)[:1200])

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

### 2.1 BM25 sparse retrieval &nbsp;&nbsp;⏳ requires GPU instance

`bm25s` over the full corpus. Indexing 1.2 M passages takes ~2-4 min.
Index is cached to `CACHE_ROOT/bm25_index/` on `/root/autodl-tmp` (NVMe, no write-rate limit).

In [ ]:
from pathlib import Path
from src.retrieval.bm25 import BM25Retriever

BM25_DIR = Path(CACHE_ROOT) / "bm25_index"   # ← autodl-tmp NVMe
if BM25_DIR.exists() and (BM25_DIR / "ev_ids.txt").exists():
    bm25 = BM25Retriever.load(BM25_DIR)
    print("BM25 loaded from cache:", BM25_DIR)
else:
    bm25 = BM25Retriever()
    bm25.build(evidence, save_dir=BM25_DIR)
    print("BM25 built and cached at", BM25_DIR)

### 2.2 Dense retrieval (BAAI/bge-m3) &nbsp;&nbsp;⏳ requires GPU instance

1024-d embeddings on the full corpus → FAISS `IndexFlatIP`.
Encoding: ~15-30 min on RTX 3090 (vs 30-60 min on T4) with `batch_size=64, fp16=True, max_seq=256`.
Chunks are written individually to autodl-tmp — no Drive rate-limit risk.
If the run is interrupted, restart this cell: already-encoded chunks are skipped automatically.

> **降级路径** (OOM 时): `max_seq_length=128` → `batch_size=16` → `model_name=LIGHT_MODEL` (bge-small-en-v1.5, 33M).

In [ ]:
from src.retrieval.dense import DenseRetriever, DEFAULT_MODEL, LIGHT_MODEL

DENSE_DIR = Path(CACHE_ROOT) / "dense_index"   # ← autodl-tmp NVMe

if DENSE_DIR.exists() and (DENSE_DIR / "faiss.index").exists():
    dense = DenseRetriever.load(DENSE_DIR, max_seq_length=256, fp16=True)
    print("dense loaded from cache:", DENSE_DIR)
else:
    # DENSE_BATCH was set by GPU-detection cell (64 for 24 GB, 32 for ≤16 GB)
    dense = DenseRetriever(
        model_name=DEFAULT_MODEL,   # "BAAI/bge-m3"  (via HF mirror if needed)
        max_seq_length=256,         # EDA: evidence 中位 18 token / max 479
        fp16=True,
    )
    dense.build(evidence, save_dir=DENSE_DIR, batch_size=DENSE_BATCH)
    print("dense built and cached at", DENSE_DIR)

### 2.3 Fusion + rerank + rule-based reorder &nbsp;&nbsp;⏳ requires GPU instance

Pipeline: BM25 top-200 + dense top-200 → weighted-sum fusion (0.3 BM25 + 0.7 dense) →
top-150 → cross-encoder rerank (`bge-reranker-base`) → top-50 →
entity-overlap boost + near-duplicate suppression → final top-`k`.

> 🧪 Pure-Python parts (`fuse.py`, `rule_reorder`) are unit-tested locally.

In [ ]:
from src.retrieval.rerank import CrossEncoderReranker
from src.retrieval.pipeline import RetrievalPipeline, RetrievalConfig

reranker = CrossEncoderReranker()   # BAAI/bge-reranker-base
pipeline_zero_shot = RetrievalPipeline(
    evidence_corpus=evidence,
    bm25=bm25,
    dense=dense,
    reranker=reranker,
    cfg=RetrievalConfig(final_k=5),
)

# Sanity: retrieve for one dev claim and compare to gold.
cid, gd = next(iter(dev.items()))
print(f"claim_id={cid}\nclaim={gd['claim_text']}\ngold={gd['evidences']}\n")
for ev_id, txt in pipeline_zero_shot.retrieve(gd["claim_text"]):
    mark = "✓" if ev_id in gd["evidences"] else " "
    print(f"  {mark} {ev_id}: {txt[:120]}")

### 2.4 k-sweep on official dev &nbsp;&nbsp;⏳ requires GPU instance

Sweep `k ∈ {1..8}` on dev (no test peeking) to find the optimal evidence count cut-off.

In [ ]:
from src.eval_helpers import recall_curve

retrieved = {}
for cid, c in dev.items():
    cands = bm25.search(c["claim_text"], k=200)
    retrieved[cid] = [eid for eid, _ in cands]

rc = recall_curve(retrieved, dev, ks=(1, 3, 5, 10, 20, 50, 100, 200))
print("BM25-only recall@k on dev:")
for k, r in rc.items():
    print(f"  k={k:>3}: recall={r:.3f}")
# Repeat with full pipeline for a richer comparison once dense + rerank are in place.

### 2.5 SFT — Qwen3.5-4B QLoRA via ms-swift &nbsp;&nbsp;⏳ requires GPU instance

**Text-only** Qwen3.5-4B base model from ModelScope (`Qwen/Qwen3.5-4B`).
LoRA on all linear layers. Training output goes to `CACHE_ROOT/sft-out/` on autodl-tmp.

| GPU | Config | Estimated time |
|---|---|---|
| RTX 4090 (24 GB) | BS=2 / GA=8 / QLoRA 4-bit | ~15-20 min/epoch |
| RTX 3090 (24 GB) | BS=2 / GA=8 / QLoRA 4-bit | ~20-25 min/epoch |
| A100 (40 GB) | BS=4 / GA=4 / bf16 | ~10-12 min/epoch |
| T4 (15 GB) | BS=1 / GA=16 / QLoRA 4-bit | ~25-35 min/epoch |

> ✅ Training data (`sft_train_v1.jsonl`, 1972 records) produced and validated locally.

In [ ]:
# Step 1 — download Qwen3.5-4B (base) from ModelScope.
# ModelScope 在国内直连，无需 VPN / HF_TOKEN。
# 模型约 8 GB，下载约 5-10 min（ModelScope CDN 在国内）。
from modelscope import snapshot_download
from pathlib import Path

MODEL_CACHE = str(Path(CACHE_ROOT) / "model_cache")   # ← autodl-tmp NVMe
MODEL_DIR = snapshot_download(
    "Qwen/Qwen3.5-4B",
    cache_dir=MODEL_CACHE,
)
print("model dir:", MODEL_DIR)

In [ ]:
# Step 2 — ms-swift SFT (参数由 GPU 检测 cell 自动设置).
from pathlib import Path

SFT_OUT   = Path(CACHE_ROOT) / "sft-out"         # ← autodl-tmp NVMe
DATA_PATH = Path(PROJECT_ROOT) / "outputs" / "sft_data" / "sft_train_v1.jsonl"
VAL_PATH  = Path(PROJECT_ROOT) / "outputs" / "sft_data" / "sft_dev_holdout_v1.jsonl"

cmd = f"""swift sft \\
  --model {MODEL_DIR} --use_hf false \\
  --train_type lora --target_modules all-linear \\
  {QUANT_FLAG} \\
  --dataset {DATA_PATH} --val_dataset {VAL_PATH} \\
  --output_dir {SFT_OUT} \\
  --num_train_epochs 3 \\
  --per_device_train_batch_size {SFT_BS} \\
  --gradient_accumulation_steps {SFT_GA} \\
  --learning_rate 2e-4 --warmup_ratio 0.03 --max_length {MAX_LEN} \\
  --gradient_checkpointing true --bf16 true \\
  --lora_rank 16 --lora_alpha 32 --lora_dropout 0.05 \\
  --save_steps 200 --eval_steps 200 \\
  --resume_from_checkpoint {SFT_OUT}/last"""

print("Effective batch size:", SFT_BS * SFT_GA)
print(cmd)
# 确认无误后取消注释：
# !{cmd}

### 2.6 DPO — preference alignment &nbsp;&nbsp;⏳ requires GPU instance

Mine wrong predictions on `dev_holdout` (never on official dev).
For each error: `(chosen=gold_response, rejected=model_response)` → ms-swift DPO.

> 🧪 Pair-construction logic in `src/dpo_pairs.py` is unit-tested locally.

In [ ]:
from pathlib import Path

DPO_OUT      = Path(CACHE_ROOT) / "dpo-out"          # ← autodl-tmp NVMe
DPO_DATA     = Path(PROJECT_ROOT) / "outputs" / "dpo_data" / "dpo_pairs.jsonl"
SFT_BEST     = Path(CACHE_ROOT) / "sft-out" / "checkpoint-final"

dpo_cmd = f"""swift rlhf --rlhf_type dpo \\
  --model {SFT_BEST} --use_hf false \\
  --train_type lora {QUANT_FLAG} \\
  --dataset {DPO_DATA} \\
  --beta 0.1 \\
  --learning_rate 5e-6 \\
  --num_train_epochs 1 \\
  --max_length {MAX_LEN} --max_prompt_length 896 \\
  --per_device_train_batch_size {SFT_BS} \\
  --gradient_accumulation_steps {SFT_GA} \\
  --output_dir {DPO_OUT} \\
  --bf16 true"""

print(dpo_cmd)
# !{dpo_cmd}

### 2.7 Self-consistency inference &nbsp;&nbsp;⏳ requires GPU instance

5 samples at T=0.7 / top_p=0.9 / max_new_tokens=32. Majority vote on label;
pick the highest-confidence sample's evidence list.

> 🧪 `src/inference.py` and `src/prompt.py` are unit-tested locally with stub retrievers.

In [ ]:
from collections import Counter
from src.prompt import build_user_query, parse_response, SYSTEM_PROMPT


def infer_one(claim_text, retrieved, model, tokenizer, n_samples=5, T=0.7):
    user = build_user_query(claim_text, retrieved)
    shown_ids = [eid for eid, _ in retrieved]
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    labels, ev_lists = [], []
    for _ in range(n_samples):
        out  = model.generate(inputs, max_new_tokens=32, do_sample=True, temperature=T, top_p=0.9)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        lbl, evs = parse_response(text, shown_ids)
        labels.append(lbl); ev_lists.append(evs)
    final_label = Counter(labels).most_common(1)[0][0]
    best = max(
        (i for i, l in enumerate(labels) if l == final_label),
        key=lambda i: len(ev_lists[i]),
    )
    return final_label, ev_lists[best]


print("infer_one() defined — wires to model + retriever once SFT done.")

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

### 3.1 Sanity-check our scorer against `eval.py` &nbsp;&nbsp;✅ verified locally

`src.eval_helpers.score_predictions` matches `eval.py` to 1e-15 on the baseline.
Per-bucket slicing added so we can split metrics by domain / scenario / difficulty on `diag_test`.

In [ ]:
import json
from pathlib import Path
from src.eval_helpers import score_predictions, score_per_bucket

with open(Path(PROJECT_ROOT) / "data" / "dev-claims-baseline.json") as f:
    baseline = json.load(f)
m = score_predictions(baseline, dev)
print(f"baseline on dev:  F={m['f_score']:.4f}  A={m['accuracy']:.4f}  HM={m['harmonic_mean']:.4f}")

### 3.2 Ablation table (computed on official dev) &nbsp;&nbsp;🧪 stub-validated locally

| ID | Config |
|---|---|
| A1 | BM25 + zero-shot Qwen3.5 |
| A2 | + dense (bge-m3) |
| A3 | + cross-encoder rerank |
| A4 | + rule reorder + HyDE |
| B1 | A4 + Qwen3.5 SFT |
| B2 | + DPO |
| B3 | + self-consistency (flagship) |
| C1 | B3 inferred with Qwen3.5-9B int4 |
| C2 | B3 without curriculum |

> `AblationHarness` in `src/ablation.py` is exercised by dry-run with stub predictions.

In [ ]:
# Table populated by running each config against dev and concatenating results.
print("Ablation harness — populated post-training.")

### 3.3 Per-domain / per-scenario / per-difficulty diagnostic (on `diag_test`) &nbsp;&nbsp;🧪 stub-validated locally

Three slice tables identify worst-performing buckets and motivate the data-centric next iteration.

In [ ]:
from src.data_io import read_jsonl
from src.paths import SPLITS_DIR

diag = list(read_jsonl(SPLITS_DIR / "diag_test.jsonl"))
diag_gold  = {r["id"]: {"claim_label": r["claim_label"], "evidences": r["evidences"]} for r in diag}
tag_lookup = {r["id"]: r for r in diag}
# Replace `predictions` with the dict produced by your pipeline on diag_test claims.
# domain_slices    = score_per_bucket(predictions, diag_gold, lambda c: tag_lookup[c]['domain'])
# scenario_slices  = score_per_bucket(predictions, diag_gold, lambda c: tag_lookup[c]['scenario'])
# difficulty_slices = score_per_bucket(predictions, diag_gold, lambda c: tag_lookup[c]['difficulty']['level'])
print(f"diag_test: {len(diag)} claims")
print("Diagnostic slices wired — populate after Stage 5 inference produces predictions.")

### 3.4 Final test predictions &nbsp;&nbsp;⏳ requires GPU instance

Reads `data/test-claims-unlabelled.json`, runs the full pipeline,
writes `outputs/test-claims-predictions.json` in `eval.py`-compatible format.
The test set is **never** inspected manually (per assignment rule).

In [ ]:
from src.data_io import write_predictions
from pathlib import Path

# preds = {cid: pipeline_predict(c['claim_text']) for cid, c in test.items()}
# write_predictions(preds, Path(PROJECT_ROOT) / "outputs" / "test-claims-predictions.json")
print("Test prediction harness — runs after final B3 model is locked.")

### 3.5 Four-track comparison (Base / Base+RAG / SFT / DPO) &nbsp;&nbsp;⏳ requires GPU instance

Isolates the contribution of each component. All four tracks share the same retrieval pipeline
(BM25 + dense + rerank) where applicable. Self-consistency (5 samples @ T=0.7) is enabled
**only on Track 4** so the SFT→DPO delta is attributable to preference alignment, not sampling.

| Track | Retrieval | Model | Decoding |
|---|---|---|---|
| 1. Base (claim only) | none | base Qwen | greedy |
| 2. Base + RAG | BM25 + dense + rerank | base Qwen | greedy |
| 3. + SFT | same as Track 2 | SFT-LoRA Qwen | greedy |
| 4. + DPO | same as Track 2 | SFT+DPO LoRA Qwen | self-consistency |

**Note on Track 1:** evidences field is a stub (`["evidence-0"]`); F-score ~0 by design.
Read its **Label Acc** in isolation as the parametric-knowledge baseline.

In [ ]:
# Cell 3.5a — load base model (used by Tracks 1 and 2)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pathlib import Path

if "MODEL_DIR" not in dir() or not Path(MODEL_DIR).exists():
    from modelscope import snapshot_download
    MODEL_DIR = snapshot_download(
        "Qwen/Qwen3.5-4B",
        cache_dir=str(Path(CACHE_ROOT) / "model_cache"),
    )
    print("downloaded base model to:", MODEL_DIR)

BASE_MODEL_DIR = MODEL_DIR

# QLoRA 4-bit — keeps VRAM footprint consistent across all GPU tiers
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
base_model.eval()
print("base model loaded:", BASE_MODEL_DIR)

In [ ]:
# Cell 3.5b — load LoRA adapters for SFT (Track 3) and DPO (Track 4)
from peft import PeftModel
from pathlib import Path

SFT_CKPT = Path(CACHE_ROOT) / "sft-out" / "checkpoint-final"
DPO_CKPT = Path(CACHE_ROOT) / "dpo-out" / "checkpoint-final"

sft_model = dpo_model = None

if SFT_CKPT.exists():
    sft_model = PeftModel.from_pretrained(base_model, str(SFT_CKPT))
    sft_model.eval()
    print("SFT adapter loaded:", SFT_CKPT)
else:
    print("SFT checkpoint not found — Track 3 skipped:", SFT_CKPT)

if DPO_CKPT.exists():
    dpo_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_DIR, quantization_config=bnb_cfg,
        device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16,
    )
    dpo_model = PeftModel.from_pretrained(dpo_base, str(DPO_CKPT))
    dpo_model.eval()
    print("DPO adapter loaded:", DPO_CKPT)
else:
    print("DPO checkpoint not found — Track 4 skipped:", DPO_CKPT)

In [ ]:
# Cell 3.5c — assemble inferers and run all 4 tracks
from src.inference import NoRagInferer, ZeroShotInferer, ModelInferer
from src.eval_compare import run_4tracks
from pathlib import Path

inferers = {
    "track1_base":     NoRagInferer(base_model, tokenizer),
    "track2_base_rag": ZeroShotInferer(pipeline_zero_shot, base_model, tokenizer),
}
if sft_model is not None:
    inferers["track3_sft"] = ZeroShotInferer(pipeline_zero_shot, sft_model, tokenizer)
if dpo_model is not None:
    inferers["track4_dpo"] = ModelInferer(
        pipeline_zero_shot, dpo_model, tokenizer,
        n_samples=5, temperature=0.7, top_p=0.9,
    )

OUT_DIR = Path(PROJECT_ROOT) / "outputs"
rows, table_path = run_4tracks(claims=dev, gold=dev, inferers=inferers, out_dir=OUT_DIR)
print("\n" + table_path.read_text(encoding="utf-8"))

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed.*

The full source lives in the `src/` package. The imports below expose the key classes
so a marker can review architecture without unzipping the resource bundle.

- `BM25Retriever` — sparse first-stage with on-disk index caching.
- `DenseRetriever` — bi-encoder + FAISS index, chunked streaming build for RAM safety.
- `CrossEncoderReranker` — second-stage scorer.
- `RetrievalPipeline` — composes the four stages and exposes a single `retrieve(claim)` method.

In [ ]:
# ── src/retrieval/bm25.py ──────────────────────────────────────────────
from src.retrieval.bm25 import BM25Retriever

In [ ]:
from src.retrieval.dense import DenseRetriever, DEFAULT_MODEL, LIGHT_MODEL

In [ ]:
from src.retrieval.rerank import CrossEncoderReranker, rule_reorder

In [ ]:
from src.retrieval.pipeline import RetrievalPipeline, RetrievalConfig